# Pulang — Wan 2.1 Image-to-Video (GRATIS, Colab)

Video pendek **Pulang** (Andra pulang setelah 7 tahun) — 3 still fotorealistik dianimasikan jadi klip bergerak pakai **Wan 2.1 I2V**, model video open-source (Apache 2.0, Alibaba). Tanpa Flow, tanpa kredit.

**JUJUR SOAL BATASANNYA — BACA DULU:**
- Wan 2.1 I2V resmi **hanya ada model 14B** (tidak ada 1.3B). Bobotnya ~40 GB (DiT 28 GB + T5 11 GB).
- Colab gratis (T4, RAM ~13 GB) kemungkinan **OOM saat load model** — notebook ini sudah pakai `--offload_model True --t5_cpu` + resolusi 480P untuk meminimalkan kebutuhan memori, tapi tetap tidak ada jaminan lolos di runtime gratis.
- Kalau OOM / gagal load: fallback yang sudah terbukti jalan di T4 adalah notebook **AniSora V3.1** (varian 12 GB) — tapi hasilnya bergaya anime, bukan fotorealistik.
- Kalau notebook ini jalan: 1 scene ~5 detik, perkiraan 10–30 menit per scene di T4 (14B itu berat).

**Cara pakai:** Runtime → Ubah jenis runtime → **T4 GPU** → jalankan sel dari atas ke bawah (Run all). Ganti `SCENE` di sel generate untuk scene lain.


In [ ]:
# 0 — Cek GPU + RAM (pastikan T4 terpilih)
import torch, subprocess, shutil
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM total: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)
print(subprocess.run(['df', '-h', '.'], capture_output=True, text=True).stdout)


In [ ]:
# 1 — Clone repo resmi Wan2.1 + install dependency
# Repo: https://github.com/Wan-Video/Wan2.1 (Apache 2.0). Install 10-20 menit.
import os, subprocess, sys
if not os.path.isdir('Wan2.1'):
    subprocess.run(['git', 'clone', 'https://github.com/Wan-Video/Wan2.1.git'], check=True)
%cd Wan2.1
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=False)
print('install selesai (cek error di atas; flash-attn kadang lama/gagal build — kalau gagal, lanjut dulu, generate bisa tetap jalan tanpa flash-attn)')


In [ ]:
# 2 — Download model I2V-14B-480P (~40 GB, 15-30 menit)
# Model: Wan-AI/Wan2.1-I2V-14B-480P (Hugging Face). 480P dipilih karena paling ringan.
# Opsi 720P: ganti MODEL='Wan-AI/Wan2.1-I2V-14B-720P' (lebih berat, lebih lambat, kualitas lebih tinggi).
import os
from huggingface_hub import snapshot_download
MODEL = 'Wan-AI/Wan2.1-I2V-14B-480P'
CKPT = snapshot_download(repo_id=MODEL,
    allow_patterns=['*.json', '*.pth', '*.safetensors', '*.txt'])
print('ckpt_dir =', CKPT)
print(sorted(os.listdir(CKPT)))


In [ ]:
# 3 — Download still Pulang (3 JPG) dari GitHub
# Fallback: upload manual pulang-stills-wan21.zip ke Colab, lalu unzip.
import os, subprocess, zipfile, urllib.request
URL = 'https://raw.githubusercontent.com/eioz21/pulang-wan21/main/pulang-stills-wan21.zip'
if not os.path.isdir('stills'):
    urllib.request.urlretrieve(URL, 'pulang-stills-wan21.zip')
    zipfile.ZipFile('pulang-stills-wan21.zip').extractall('stills')
print(sorted(os.listdir('stills')))
for f in sorted(os.listdir('stills')):
    from PIL import Image
    im = Image.open(os.path.join('stills', f)); print(f, im.size)


In [ ]:
# 4 — Scene Pulang: prompt gerak (Inggris, seperti contoh resmi README)
SCENES = {
    'scene01': {
        'image': 'stills/scene01.jpg',
        'prompt': 'The young man sitting by the bus window slowly turns his head toward the window and gazes at the passing city lights, quiet and thoughtful. Subtle natural movement, gentle slow camera push-in, cinematic, photorealistic, no text.',
    },
    'scene02': {
        'image': 'stills/scene02.jpg',
        'prompt': 'The young man with a backpack walks slowly away from the camera down the misty village road toward the warm glowing house in the distance. Mist drifts gently, puddles reflect the light, camera follows slowly, cinematic, photorealistic, no text.',
    },
    'scene03': {
        'image': 'stills/scene03.jpg',
        'prompt': 'The elderly mother opens the wooden front door and smiles warmly at her son standing on the doorstep, tears of joy, warm yellow light spilling out. Both move slightly as they greet each other, gentle and emotional, cinematic, photorealistic, no text.',
    },
}
# >>> GANTI scene di sini untuk generate scene lain: 'scene01' / 'scene02' / 'scene03'
SCENE = 'scene01'
SIZE = '832*480'      # 480P (paling ringan). 720P: '1280*720' + model 720P
FRAME_NUM = 81         # 81 frame @16fps = ~5 detik (harus 4n+1)
SEED = 4096
print(SCENE, '->', SCENES[SCENE]['image'])
print(SCENES[SCENE]['prompt'])


In [ ]:
# 5 — Generate scene (PERINTAH RESMI README Wan2.1, tanpa prompt extension)
# 14B di 1 GPU: --offload_model True --t5_cpu wajib. sample_steps default 40 untuk I2V,
# sample_shift default 3.0 untuk 832*480. Estimasi 10-30 menit/scene di T4.
import os, subprocess, sys
s = SCENES[SCENE]
os.makedirs('outputs', exist_ok=True)
out = f'outputs/pulang-{SCENE}.mp4'
cmd = [sys.executable, 'generate.py', '--task', 'i2v-14B', '--size', SIZE,
       '--ckpt_dir', CKPT, '--image', s['image'], '--prompt', s['prompt'],
       '--frame_num', str(FRAME_NUM), '--base_seed', str(SEED),
       '--offload_model', 'True', '--t5_cpu',
       '--save_file', out]
print(' '.join(cmd))
r = subprocess.run(cmd)
print('returncode:', r.returncode)
print('output ada:', os.path.exists(out), out if os.path.exists(out) else '')


In [ ]:
# 6 — Lihat + download hasil
import os, glob
from IPython.display import Video, display
for f in sorted(glob.glob('outputs/*.mp4')):
    print(f, os.path.getsize(f) // 1024, 'KB')
    display(Video(f, embed=True, width=640))
print('Download: klik kanan file di panel Files (outputs/) > Download, atau:')
print('from google.colab import files; files.download("outputs/pulang-%s.mp4" % SCENE)')


## Kalau gagal

- **CUDA out of memory / RAM habis saat load:** 14B tidak muat di runtime gratis. Coba restart runtime (Runtime → Restart runtime) lalu jalankan ulang dari sel 2, atau pakai runtime RAM besar (Colab Pro). Fallback gratis yang sudah terbukti: notebook AniSora V3.1 di https://github.com/eioz21/btl-anisora (hasil bergaya anime).
- **flash-attn gagal build:** abaikan, generate tetap bisa jalan (lebih lambat).
- **Hasilnya diam / gerak minim:** naikkan FRAME_NUM tetap 81, coba ubah prompt geraknya jadi lebih eksplisit (mis. 'walks forward', 'turns his head'), atau ganti SEED.
- **720P:** ganti MODEL di sel 2 ke `Wan-AI/Wan2.1-I2V-14B-720P` dan SIZE di sel 4 ke `'1280*720'`. Jauh lebih berat — hanya kalau 480P sudah jalan.

## Setelah 3 scene jadi

Download ketiga mp4, kirim ke Mamah — nanti dijahit jadi satu video Pulang + narasi + subtitle + musik, seperti video Pulang v1.
